In [ ]:
# ==========================================
# 1. Environment Setup & Imports
# ==========================================
%load_ext autoreload
%autoreload 2

import sys
import gc
from pathlib import Path
import torch
import pandas as pd
import numpy as np

# Import Shared Utilities
current_dir = Path.cwd()
if str(current_dir) not in sys.path:
    sys.path.append(str(current_dir))

try:
    import analysis_utils as utils
    from analysis_utils import PROJECT_ROOT
    print(f"Project Root: {PROJECT_ROOT}")
except ImportError as e:
    print(f"Failed to import analysis_utils: {e}")

In [ ]:
# ==========================================
# 2. Configuration
# ==========================================
# Experiment Settings
EXP_PREFIX = 'Exp5_ViT'     
DATA_PREFIX = 'Exp5_'       
MODEL_PATH_OVERRIDE = None 

# Paths
RESULTS_DIR = PROJECT_ROOT / 'Results'
CACHE_DIR   = PROJECT_ROOT / '.cache'

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Plotting
FIG_DIR_NAME = 'figures'

In [ ]:
# ==========================================
# 3. Load Data & Model
# ==========================================
# A. Load Cached Splits
splits = ['train', 'val', 'test']
datasets = {}
print("Loading cached datasets...")
for split in splits:
    pt_path = CACHE_DIR / f"{DATA_PREFIX}{split}.pt"
    if pt_path.exists():
        try:
            ds = utils.CachedPTDataset(pt_path)
            datasets[split] = ds
            print(f"  - [{split.upper()}] Loaded {len(ds)} samples ({ds.kind})")
        except Exception as e:
            print(f"  - [{split.upper()}] Failed to load: {e}")
            datasets[split] = None
    else:
        print(f"  - [{split.upper()}] Not found: {pt_path.name}")
        datasets[split] = None

# B. Find & Load Model
model_path = Path(MODEL_PATH_OVERRIDE) if MODEL_PATH_OVERRIDE else utils.find_latest_model(RESULTS_DIR, EXP_PREFIX)
if not model_path:
    raise RuntimeError(f"No model found for {EXP_PREFIX}")

print(f"Target Model: {model_path}")
model, model_type = utils.load_analysis_model(model_path, DEVICE, datasets)
print(f"Model Type: {model_type}")

# Setup Output Dir
output_dir = model_path.parent
fig_dir = output_dir / FIG_DIR_NAME
fig_dir.mkdir(parents=True, exist_ok=True)

# Data Save Dir for H5
data_save_dir = model_path.parent / 'prediction_data'
data_save_dir.mkdir(parents=True, exist_ok=True)
print(f"Prediction H5 data will be saved to: {data_save_dir}")

In [ ]:
# ==========================================
# 4. Execution: Predictions
# ==========================================
all_results = {}

# A. Cached Test Set (In-Distribution)
if datasets['test']:
    print(f"Predicting on Cached Test Set (In-Dist)...")
    df_test, feats, x_data = utils.predict_dataset(
        datasets['test'], model, DEVICE, model_type, return_feats=True
    )
    df_test['type'] = 'Cached In-Dist'
    all_results['test_cached'] = df_test
    
    # Save H5
    utils.save_to_h5(
        data_save_dir / f"{EXP_PREFIX}test_cached_indist.h5", 
        df_test, feats, x_data, 'test_cached_indist'
    )
    del feats, x_data; gc.collect()

# B. Cached Out-Of-Distribution (Held Out)
out_pt_path = CACHE_DIR / f"{DATA_PREFIX}out.pt"
if out_pt_path.exists():
    print(f"Predicting on Held-Out Test Set (Out-Dist): {out_pt_path.name}...")
    try:
        ds_out = utils.CachedPTDataset(out_pt_path)
        df_out, feats, x_data = utils.predict_dataset(
            ds_out, model, DEVICE, model_type, return_feats=True
        )
        df_out['type'] = 'Cached Out-Dist'
        all_results['test_out'] = df_out
        
        # Save H5
        utils.save_to_h5(
            data_save_dir / f"{EXP_PREFIX}test_cached_outdist.h5", 
            df_out, feats, x_data, 'test_cached_outdist'
        )
        del feats, x_data; gc.collect()
    except Exception as e:
        print(f"  [Error] Failed to load Out-Dist: {e}")

# C. All Raw Datasets
list_dg = [
    'disk_galaxy_fiducial', 'disk_galaxy_supplement', 
    'disk_galaxy_fiducial_4', 'disk_galaxy_fiducial_7', 
    'disk_galaxy_fiducial_10', 'disk_galaxy_low'
]

raw_paths = []
if utils.Exp_dg_folder_paths:
    raw_paths = utils.Exp_dg_folder_paths
else:
    # Manual Fallback
    res = 'coarse'
    for fname in list_dg:
        p = PROJECT_ROOT / 'Data' / fname / res
        if p.exists():
            raw_paths.append((str(p), np.array([0,0])))

print(f"Processing {len(raw_paths)} Raw Datasets...")
dfs_raw = []
for p_conf in raw_paths:
    try:
        ds_raw = utils.HDF5Dataset(p_conf)
        print(f"  > {ds_raw.type} ({len(ds_raw)})")
        
        df, feats, x_data = utils.predict_dataset(
            ds_raw, model, DEVICE, model_type, return_feats=True
        )
        
        # Save H5
        utils.save_to_h5(
            data_save_dir / f"{ds_raw.type}.h5", 
            df, feats, x_data, ds_raw.type
        )
        
        dfs_raw.append(df)
        del feats, x_data; gc.collect()
    except Exception as e:
        print(f"  [Error] {p_conf}: {e}")

if dfs_raw:
    all_results['raw_all'] = pd.concat(dfs_raw, ignore_index=True)

# Save predictions as CSV (for quick inspection)
for k, df in all_results.items():
    s_path = output_dir / f"{k}_predictions.csv"
    df.to_csv(s_path, index=False)
    print(f"Saved: {s_path.name}")

In [ ]:
# ==========================================
# 5. Visualization & Metrics
# ==========================================
metrics_list = []

# 1. Cached Test (In-Dist)
if 'test_cached' in all_results:
    df = all_results['test_cached']
    metrics_list.append(utils.get_metrics_summary(df['y_true'], df['y_pred'], "Cached In-Dist"))
    utils.plot_scatter(df, "Cached In-Dist", fig_dir)

# 2. Cached Test (Out-Dist)
if 'test_out' in all_results:
    df = all_results['test_out']
    metrics_list.append(utils.get_metrics_summary(df['y_true'], df['y_pred'], "Cached Out-Dist"))
    utils.plot_scatter(df, "Cached Out-Dist", fig_dir)

# 3. Raw Data
if 'raw_all' in all_results:
    df_raw = all_results['raw_all']
    metrics_list.append(utils.get_metrics_summary(df_raw['y_true'], df_raw['y_pred'], "Raw All (Model)"))
    
    if 'y_bondi' in df_raw.columns:
         valid_b = df_raw.dropna(subset=['y_bondi'])
         metrics_list.append(utils.get_metrics_summary(valid_b['y_true'], valid_b['y_bondi'], "Raw All (Bondi)"))

    utils.plot_scatter(df_raw, "All Raw Datasets", fig_dir)
    utils.plot_error_dist(df_raw, "Raw Datasets", fig_dir)
    utils.plot_kde_contour(df_raw, "Raw Datasets", fig_dir)
    
    # Per Galaxy Type
    for t in df_raw['type'].unique():
        sub = df_raw[df_raw['type'] == t]
        metrics_list.append(utils.get_metrics_summary(sub['y_true'], sub['y_pred'], f"Type: {t}"))
        utils.plot_mass_evolution(sub, t, fig_dir)

# 4. Summary Table
metrics_df = pd.DataFrame(metrics_list)
pd.options.display.float_format = '{:,.3f}'.format
display(metrics_df)
metrics_df.to_csv(output_dir / "summary_metrics.csv", index=False)

In [ ]:
# ==========================================
# 6. Save Markdown Report
# ==========================================
md_path = output_dir / "summary_metrics.md"
with open(md_path, 'w', encoding='utf-8') as f:
    f.write(f"# Analysis Results: {EXP_PREFIX}\n\n")
    f.write(f"- **Model**: `{model_path.name}`\n")
    f.write(f"- **Type**: `{model_type}`\n\n")
    
    f.write("## Metrics\n")
    f.write(metrics_df.to_markdown(floatfmt=".3f"))
    
    f.write("\n\n## Visualizations\n")
    for img in sorted(fig_dir.glob(f"*.png")):
        f.write(f"### {img.stem}\n![{img.stem}]({FIG_DIR_NAME}/{img.name})\n\n")
        
print(f"Saved report to {md_path}")